# Saluki for Gluc 3′ UTR design

[Notebook](saluki-gluc-3utr.ipynb) · [Installation](../docs/installation.md)

Optimize a 150-nt Gluc 3′ UTR for predicted mRNA stability. Keep the HBB 5′ UTR and Gluc CDS fixed.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, random_seed_sequences, select_candidates

utr5 = "ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC"
cds = (
    "ATGGGAGTCAAAGTTCTGTTTGCCCTGATCTGCATCGCTGTGGCCGAGGCCAAGCCCACCGAGAACAACGAA"
    "GACTTCAACATCGTGGCCGTGGCCAGCAACTTCGCGACCACGGATCTCGATGCTGACCGCGGGAAGTTGCCC"
    "GGCAAGAAGCTGCCGCTGGAGGTGCTCAAAGAGATGGAAGCCAATGCCCGGAAAGCTGGCTGCACCAGGGGC"
    "TGTCTGATCTGCCTGTCCCACATCAAGTGCACGCCCAAGATGAAGAAGTTCATCCCAGGACGCTGCCACACC"
    "TACGAAGGCGACAAAGAGTCCGCACAGGGCGGCATAGGCGAGGCGATCGTCGACATTCCTGAGATTCCTGGG"
    "TTCAAGGACTTGGAGCCCATGGAGCAGTTCATCGCACAGGTCGATCTGTGTGTGGACTGCACAACTGGCTGC"
    "CTCAAAGGGCTTGCCAACGTGCAGTGTTCTGACCTGCTCAAGAAGTGGCTGCCGCAACGCTGTGCGACCTTT"
    "GCCAGCAAGATCCAGGGCCAGGTGGACAAGATCAAGGGGGCCGGTGGTGACTAA"
)
length = 150
prefix = utr5 + cds

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
saluki = load_model(
    "saluki", device=device,
    cds_start=len(utr5), cds_end=len(prefix),
    exon_lengths=[len(prefix) + length],
)

indices = torch.tensor(["ACGT".index(base) for base in prefix])
fixed = torch.nn.functional.one_hot(indices, num_classes=4)


class UTRScorer:
    task_names = saluki.task_names
    device = saluki.device

    def score(self, sequences):
        return saluki.score([prefix + sequence for sequence in sequences])

    def score_soft(self, probabilities):
        fixed_batch = fixed.to(probabilities).expand(len(probabilities), -1, -1)
        transcripts = torch.cat([fixed_batch, probabilities], dim=1)
        return saluki.score_soft(transcripts)


model = UTRScorer()
run = Path("outputs/saluki_gluc_3utr")




## 2. Generate candidates



In [2]:
seeds = random_seed_sequences(length, 100, seed=42)
designs = generate_designs(
    model, length=length, initial=seeds, objective="maximize",
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,0.968337,1.001676
basesearch,100,0.923713,0.978642
beam_search,100,1.017275,1.089253
cross_entropy_method,100,0.925680,0.980720
den,100,0.985943,1.011349
fast_seqprop,100,0.965055,1.011964
genetic_algorithm,100,0.969404,0.999110
ism_greedy,100,0.907591,1.009199
ledidi,100,0.817342,0.935514




## 3. Discover motifs



In [3]:
motifs = call_motifs(designs, model, reverse_complement=False, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-UYAAUAWWD,591,558,1158.0
1,2-CCCWAUWAR,281,240,386.0
4,6-UAAAUAAGY,221,217,31.0
5,7-CUAUAUGUAG,107,101,56.0
6,9-GUACCCCUAAUU,138,136,270.0
9,19-UAUWGCYCC,30,30,61.0
11,21-CAAUAAUKUUA,137,136,33.0
14,28-UCCCGCGC,12,12,232.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,CTAATAAATATCTTTTACTGAGTTTACCCCCGCCCTGTGCTCGCCC...,1.089253,1-UYAAUAWWD;28-UCCCGCGC
513,2,TATTAATGTAGTTCCCCCCCCCCCCCCGTGCCCCGCTATATTCCTC...,0.961360,28-UCCCGCGC
583,3,AGTATCATATCTCTATAAATCCCCCCCCCTATAGCCCCTTTCCCAC...,0.957485,19-UAUWGCYCC
684,4,TAGTTCCCTCTATAAGTAAATACTAAATTAATCGATTTGCACGCCG...,0.934669,7-CUAUAUGUAG
571,5,CCCCTATTCTTAGCTAACATCCCTTTATGCCTTCCCCCCCCATTCG...,0.958426,9-GUACCCCUAAUU
454,6,CCTCTATTATTGGTTATCTCCCCTCTATATCTCCTCCCCCCATTCC...,0.964593,2-CCCWAUWAR
459,7,TCCCCCCACGCCCCCCCCCCCCCCACAATATAAATACGCTATTATC...,0.964450,6-UAAAUAAGY
665,8,TTAATATAGCCGTTGATGGGTTGGGCCGACGTGCCGGTGCGTTGCC...,0.939193,1-UYAAUAWWD;2-CCCWAUWAR;7-CUAUAUGUAG;9-GUACCCC...
828,9,TATCCGGGCGCAGGCAACAATACAATAAAGTTGGCGTCCCTAACCC...,0.911894,21-CAAUAAUKUUA
858,10,CACCTTTATAGTCCCTGCCCCTCTTAGTGTTATAAGATCCCGCGTG...,0.903914,1-UYAAUAWWD;19-UAUWGCYCC;21-CAAUAAUKUUA
